# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name:Yufei Qin 

Date: 08/19/2026

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
!pip install requests
!pip install yfinance
!pip install python-dotenv
!pip install beautifulsoup4

  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 21.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 34.6 MB/s  0:00:00
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)

   ----- ---------------------------------- 1/7 [websockets]
   ----- ---------------------------------- 1/7 [websockets]
   ----- ---------------------------------- 1/7 [websockets]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------------- ---------------------- 3/7 [peewee]
   ----------------- ---------------------- 3/7 [peewee]
   ---------------------- ----------------- 4/7 [lxml

In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: c:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework04\notebooks

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [1]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [2]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [4]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

[*********************100%***********************]  1 of 1 completed


{'missing': [], 'shape': (64, 2), 'na_total': 0}

In [5]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data\raw\api_source-yfinance_symbol-AAPL_20260820-105634.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [6]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {
    'User-Agent': 'AFE-Homework/1.0'
}

resp = requests.get(
    SCRAPE_URL,
    headers=headers,
    timeout=30
)

resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')

table = soup.find('table')

rows = [
    [c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
    for tr in table.find_all('tr')
]

rows = [r for r in rows if r]

header = rows[0]
data = rows[1:]

df_scrape = pd.DataFrame(data, columns=header)

print("Shape:", df_scrape.shape)
print("Columns:", df_scrape.columns.tolist())

df_scrape.head()

Shape: (503, 8)
Columns: ['Symbol', 'Security', 'GICSSector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [8]:
required_columns = [
    'Symbol',
    'Security',
    'GICSSector'
]

v_scrape = validate(df_scrape, required_columns)

print(v_scrape)

assert len(v_scrape['missing']) == 0
assert df_scrape.shape[0] > 0

print("Scraping validation passed.")

{'missing': [], 'shape': (503, 8), 'na_total': 0}
Scraping validation passed.


In [9]:
_ = save_csv(
    df_scrape,
    prefix='scrape',
    site='wikipedia',
    table='sp500'
)

Saved data\raw\scrape_site-wikipedia_table-sp500_20260820-105821.csv


## Documentation
- API Source: (URL/endpoint/params)
- Scrape Source: (URL/table description)
- Assumptions & risks: (rate limits, selector fragility, schema changes)
- Confirm `.env` is not committed.